In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split

In [3]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_data_path = project_root / "data" / "processed"

feature_dataset_path = processed_data_path / "customer_features.csv"

In [4]:
print("Feature dataset path:", feature_dataset_path)
print("File exists:", feature_dataset_path.exists())

Feature dataset path: c:\Users\tOBESky\Documents\ML_BigDATA\saas-customer-churn-prediction\data\processed\customer_features.csv
File exists: True


In [5]:
customer_features = pd.read_csv(feature_dataset_path)

print("Dataset shape:", customer_features.shape)

customer_features.head()

Dataset shape: (500, 41)


,account_id,industry,country,signup_date,referral_source,initial_plan_tier,initial_seats,churn_flag,subscription_count,ever_upgraded,...,average_resolution_hours,average_satisfaction_score,escalation_count,last_support_ticket_date,has_support_tickets,customer_tenure_days,seat_change,plan_changed,days_since_last_support_ticket,has_satisfaction_score
0,A-2e4581,EdTech,US,2024-10-16,partner,Basic,9,0,10,1,...,23.000000,3.000000,0.0,2024-12-10,1,76,35,0,21.0,1
1,A-43a9e3,FinTech,IN,2023-08-17,other,Basic,18,1,8,1,...,38.000000,4.000000,0.0,2024-06-26,1,502,0,1,188.0,1
2,A-0a282f,DevTools,US,2024-08-27,organic,Basic,1,0,15,1,...,43.666667,4.666667,0.0,2024-10-26,1,126,1,1,66.0,1
3,A-1f0ac7,HealthTech,UK,2023-08-27,other,Basic,24,0,7,1,...,29.000000,NaN,0.0,2024-04-12,1,492,0,1,263.0,0
4,A-ce550d,HealthTech,US,2024-10-27,event,Enterprise,35,1,9,1,...,42.285714,3.800000,1.0,2024-10-25,1,65,74,0,67.0,1


In [6]:
print("Total rows:", len(customer_features))
print("Unique accounts:", customer_features["account_id"].nunique())

print("\nTarget distribution:")
print(customer_features["churn_flag"].value_counts())

print("\nTarget percentage:")
print(
    customer_features["churn_flag"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nhas_satisfaction_score exists:")
print("has_satisfaction_score" in customer_features.columns)

Total rows: 500
Unique accounts: 500

Target distribution:
churn_flag
0    390
1    110
Name: count, dtype: int64

Target percentage:
churn_flag
0    78.0
1    22.0
Name: proportion, dtype: float64

has_satisfaction_score exists:
True


In [7]:
feature_columns = [
    # Account characteristics
    "industry",
    "country",
    "referral_source",
    "initial_plan_tier",
    "customer_tenure_days",

    # Subscription behaviour
    "subscription_count",
    "ever_upgraded",
    "ever_downgraded",
    "latest_plan_tier",
    "latest_seats",
    "seat_change",
    "latest_mrr",
    "latest_is_trial",
    "latest_billing_frequency",
    "latest_auto_renew",
    "plan_changed",

    # Product usage
    "usage_event_count",
    "total_usage_count",
    "unique_features_used",
    "active_usage_days",
    "beta_usage_events",
    "errors_per_100_usage",
    "average_duration_per_event",
    "average_usage_per_event",
    "days_since_last_usage",

    # Support behaviour
    "ticket_count",
    "has_support_tickets",
    "average_first_response_minutes",
    "average_resolution_hours",
    "average_satisfaction_score",
    "has_satisfaction_score",
    "escalation_count",
    "days_since_last_support_ticket"
]

X = customer_features[feature_columns].copy()
y = customer_features["churn_flag"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (500, 33)
y shape: (500,)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTraining percentage:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTest target distribution:")
print(y_test.value_counts())

print("\nTest percentage:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

X_train shape: (400, 33)
X_test shape: (100, 33)

Training target distribution:
churn_flag
0    312
1     88
Name: count, dtype: int64

Training percentage:
churn_flag
0    78.0
1    22.0
Name: proportion, dtype: float64

Test target distribution:
churn_flag
0    78
1    22
Name: count, dtype: int64

Test percentage:
churn_flag
0    78.0
1    22.0
Name: proportion, dtype: float64


In [10]:
training_missing_values = (
    X_train.isna()
    .sum()
    .sort_values(ascending=False)
)

training_missing_values[
    training_missing_values > 0
]

average_satisfaction_score        31
average_first_response_minutes     8
days_since_last_support_ticket     8
average_resolution_hours           8
dtype: int64

In [11]:
categorical_features = [
    "industry",
    "country",
    "referral_source",
    "initial_plan_tier",
    "latest_plan_tier",
    "latest_billing_frequency"
]

numeric_features = [
    column
    for column in X_train.columns
    if column not in categorical_features
]

print("Categorical features:", len(categorical_features))
print("Numeric features:", len(numeric_features))

print("\nCategorical columns:")
print(categorical_features)

print("\nNumeric columns:")
print(numeric_features)

Categorical features: 6
Numeric features: 27

Categorical columns:
['industry', 'country', 'referral_source', 'initial_plan_tier', 'latest_plan_tier', 'latest_billing_frequency']

Numeric columns:
['customer_tenure_days', 'subscription_count', 'ever_upgraded', 'ever_downgraded', 'latest_seats', 'seat_change', 'latest_mrr', 'latest_is_trial', 'latest_auto_renew', 'plan_changed', 'usage_event_count', 'total_usage_count', 'unique_features_used', 'active_usage_days', 'beta_usage_events', 'errors_per_100_usage', 'average_duration_per_event', 'average_usage_per_event', 'days_since_last_usage', 'ticket_count', 'has_support_tickets', 'average_first_response_minutes', 'average_resolution_hours', 'average_satisfaction_score', 'has_satisfaction_score', 'escalation_count', 'days_since_last_support_ticket']


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [13]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [14]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

## Missing Value and Categorical Preprocessing Strategy

The engineered customer dataset intentionally retains some missing values when the absence itself has business meaning.

For modelling:

- numeric features are imputed using the training-set median;
- categorical features are imputed using the most frequent training category;
- categorical variables are converted to numeric features using one-hot encoding;
- numeric features are standardised for Logistic Regression.

These transformations are contained inside a scikit-learn `Pipeline` and `ColumnTransformer`, ensuring that preprocessing is fitted only on the training data and applied consistently to unseen data.fit

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [16]:
from sklearn.linear_model import LogisticRegression


logistic_regression_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

In [17]:
logistic_regression_pipeline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](33,)","['industry','country','referral_source',...,'has_satisfaction_score', 'escalation_count','days_since_last_support_ticket']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,33
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default

In [18]:
y_test_predictions = logistic_regression_pipeline.predict(
    X_test
)

In [19]:
print("First 10 predictions:")
print(y_test_predictions[:10])

print("\nFirst 10 actual values:")
print(y_test.iloc[:10].values)

First 10 predictions:
[0 0 0 0 0 0 0 0 0 0]

First 10 actual values:
[0 0 0 0 0 0 0 0 0 1]


In [20]:
y_test_probabilities = logistic_regression_pipeline.predict_proba(
    X_test
)[:, 1]

In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [22]:
accuracy = accuracy_score(
    y_test,
    y_test_predictions
)

precision = precision_score(
    y_test,
    y_test_predictions,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_test_predictions,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_test_predictions,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_test_probabilities
)

print("Accuracy:", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1-score:", round(f1, 3))
print("ROC-AUC:", round(roc_auc, 3))

Accuracy: 0.77
Precision: 0.4
Recall: 0.091
F1-score: 0.148
ROC-AUC: 0.603


In [23]:
confusion_matrix(
    y_test,
    y_test_predictions
)

array([[75,  3],
       [20,  2]])

In [24]:
balanced_logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

In [25]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [26]:
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [27]:
scoring_metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

In [28]:
baseline_cv_results = cross_validate(
    logistic_regression_pipeline,
    X_train,
    y_train,
    cv=cross_validation,
    scoring=scoring_metrics
)

In [29]:
print("Baseline Logistic Regression")

print(
    "Accuracy:",
    round(baseline_cv_results["test_accuracy"].mean(), 3)
)

print(
    "Precision:",
    round(baseline_cv_results["test_precision"].mean(), 3)
)

print(
    "Recall:",
    round(baseline_cv_results["test_recall"].mean(), 3)
)

print(
    "F1:",
    round(baseline_cv_results["test_f1"].mean(), 3)
)

print(
    "ROC-AUC:",
    round(baseline_cv_results["test_roc_auc"].mean(), 3)
)

Baseline Logistic Regression
Accuracy: 0.747
Precision: 0.255
Recall: 0.08
F1: 0.119
ROC-AUC: 0.474


In [30]:
balanced_cv_results = cross_validate(
    balanced_logistic_pipeline,
    X_train,
    y_train,
    cv=cross_validation,
    scoring=scoring_metrics
)

In [31]:
print("Balanced Logistic Regression")

print(
    "Accuracy:",
    round(balanced_cv_results["test_accuracy"].mean(), 3)
)

print(
    "Precision:",
    round(balanced_cv_results["test_precision"].mean(), 3)
)

print(
    "Recall:",
    round(balanced_cv_results["test_recall"].mean(), 3)
)

print(
    "F1:",
    round(balanced_cv_results["test_f1"].mean(), 3)
)

print(
    "ROC-AUC:",
    round(balanced_cv_results["test_roc_auc"].mean(), 3)
)

Balanced Logistic Regression
Accuracy: 0.545
Precision: 0.21
Recall: 0.387
F1: 0.271
ROC-AUC: 0.484


In [32]:
from sklearn.ensemble import RandomForestClassifier

In [33]:
random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

In [34]:
random_forest_cv_results = cross_validate(
    random_forest_pipeline,
    X_train,
    y_train,
    cv=cross_validation,
    scoring=scoring_metrics
)

In [35]:
print("Random Forest")

print(
    "Accuracy:",
    round(random_forest_cv_results["test_accuracy"].mean(), 3)
)

print(
    "Precision:",
    round(random_forest_cv_results["test_precision"].mean(), 3)
)

print(
    "Recall:",
    round(random_forest_cv_results["test_recall"].mean(), 3)
)

print(
    "F1:",
    round(random_forest_cv_results["test_f1"].mean(), 3)
)

print(
    "ROC-AUC:",
    round(random_forest_cv_results["test_roc_auc"].mean(), 3)
)

Random Forest
Accuracy: 0.745
Precision: 0.254
Recall: 0.08
F1: 0.119
ROC-AUC: 0.413


In [36]:
model_comparison = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "Balanced Logistic Regression",
            "Random Forest"
        ],
        "Accuracy": [
            baseline_cv_results["test_accuracy"].mean(),
            balanced_cv_results["test_accuracy"].mean(),
            random_forest_cv_results["test_accuracy"].mean()
        ],
        "Precision": [
            baseline_cv_results["test_precision"].mean(),
            balanced_cv_results["test_precision"].mean(),
            random_forest_cv_results["test_precision"].mean()
        ],
        "Recall": [
            baseline_cv_results["test_recall"].mean(),
            balanced_cv_results["test_recall"].mean(),
            random_forest_cv_results["test_recall"].mean()
        ],
        "F1": [
            baseline_cv_results["test_f1"].mean(),
            balanced_cv_results["test_f1"].mean(),
            random_forest_cv_results["test_f1"].mean()
        ],
        "ROC_AUC": [
            baseline_cv_results["test_roc_auc"].mean(),
            balanced_cv_results["test_roc_auc"].mean(),
            random_forest_cv_results["test_roc_auc"].mean()
        ]
    }
)

model_comparison.round(3)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.747,0.255,0.080,0.119,0.474
1,Balanced Logistic Regression,0.545,0.210,0.387,0.271,0.484
2,Random Forest,0.745,0.254,0.080,0.119,0.413


In [37]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [38]:
from xgboost import XGBClassifier

In [39]:
retained_customers = (y_train == 0).sum()
churned_customers = (y_train == 1).sum()

scale_pos_weight = retained_customers / churned_customers

print("Retained customers:", retained_customers)
print("Churned customers:", churned_customers)
print("Scale positive weight:", round(scale_pos_weight, 2))

Retained customers: 312
Churned customers: 88
Scale positive weight: 3.55


In [40]:
xgboost_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBClassifier(
                n_estimators=200,
                max_depth=3,
                learning_rate=0.05,
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

In [41]:
xgboost_cv_results = cross_validate(
    xgboost_pipeline,
    X_train,
    y_train,
    cv=cross_validation,
    scoring=scoring_metrics
)

In [42]:
print("XGBoost")

print(
    "Accuracy:",
    round(xgboost_cv_results["test_accuracy"].mean(), 3)
)

print(
    "Precision:",
    round(xgboost_cv_results["test_precision"].mean(), 3)
)

print(
    "Recall:",
    round(xgboost_cv_results["test_recall"].mean(), 3)
)

print(
    "F1:",
    round(xgboost_cv_results["test_f1"].mean(), 3)
)

print(
    "ROC-AUC:",
    round(xgboost_cv_results["test_roc_auc"].mean(), 3)
)

XGBoost
Accuracy: 0.662
Precision: 0.193
Recall: 0.171
F1: 0.178
ROC-AUC: 0.455


In [43]:
model_comparison = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "Balanced Logistic Regression",
            "Random Forest",
            "XGBoost"
        ],
        "Accuracy": [
            baseline_cv_results["test_accuracy"].mean(),
            balanced_cv_results["test_accuracy"].mean(),
            random_forest_cv_results["test_accuracy"].mean(),
            xgboost_cv_results["test_accuracy"].mean()
        ],
        "Precision": [
            baseline_cv_results["test_precision"].mean(),
            balanced_cv_results["test_precision"].mean(),
            random_forest_cv_results["test_precision"].mean(),
            xgboost_cv_results["test_precision"].mean()
        ],
        "Recall": [
            baseline_cv_results["test_recall"].mean(),
            balanced_cv_results["test_recall"].mean(),
            random_forest_cv_results["test_recall"].mean(),
            xgboost_cv_results["test_recall"].mean()
        ],
        "F1": [
            baseline_cv_results["test_f1"].mean(),
            balanced_cv_results["test_f1"].mean(),
            random_forest_cv_results["test_f1"].mean(),
            xgboost_cv_results["test_f1"].mean()
        ],
        "ROC_AUC": [
            baseline_cv_results["test_roc_auc"].mean(),
            balanced_cv_results["test_roc_auc"].mean(),
            random_forest_cv_results["test_roc_auc"].mean(),
            xgboost_cv_results["test_roc_auc"].mean()
        ]
    }
)

model_comparison.round(3)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.747,0.255,0.080,0.119,0.474
1,Balanced Logistic Regression,0.545,0.210,0.387,0.271,0.484
2,Random Forest,0.745,0.254,0.080,0.119,0.413
3,XGBoost,0.662,0.193,0.171,0.178,0.455


In [44]:
cv_stability = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "Balanced Logistic Regression",
            "Random Forest",
            "XGBoost"
        ],

        "Recall Mean": [
            baseline_cv_results["test_recall"].mean(),
            balanced_cv_results["test_recall"].mean(),
            random_forest_cv_results["test_recall"].mean(),
            xgboost_cv_results["test_recall"].mean()
        ],

        "Recall Std": [
            baseline_cv_results["test_recall"].std(),
            balanced_cv_results["test_recall"].std(),
            random_forest_cv_results["test_recall"].std(),
            xgboost_cv_results["test_recall"].std()
        ],

        "F1 Mean": [
            baseline_cv_results["test_f1"].mean(),
            balanced_cv_results["test_f1"].mean(),
            random_forest_cv_results["test_f1"].mean(),
            xgboost_cv_results["test_f1"].mean()
        ],

        "F1 Std": [
            baseline_cv_results["test_f1"].std(),
            balanced_cv_results["test_f1"].std(),
            random_forest_cv_results["test_f1"].std(),
            xgboost_cv_results["test_f1"].std()
        ],

        "ROC_AUC Mean": [
            baseline_cv_results["test_roc_auc"].mean(),
            balanced_cv_results["test_roc_auc"].mean(),
            random_forest_cv_results["test_roc_auc"].mean(),
            xgboost_cv_results["test_roc_auc"].mean()
        ],

        "ROC_AUC Std": [
            baseline_cv_results["test_roc_auc"].std(),
            balanced_cv_results["test_roc_auc"].std(),
            random_forest_cv_results["test_roc_auc"].std(),
            xgboost_cv_results["test_roc_auc"].std()
        ]
    }
)

cv_stability.round(3)

,Model,Recall Mean,Recall Std,F1 Mean,F1 Std,ROC_AUC Mean,ROC_AUC Std
0,Logistic Regression,0.080,0.048,0.119,0.060,0.474,0.048
1,Balanced Logistic Regression,0.387,0.068,0.271,0.038,0.484,0.045
2,Random Forest,0.080,0.028,0.119,0.037,0.413,0.048
3,XGBoost,0.171,0.075,0.178,0.061,0.455,0.038


In [45]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve
import numpy as np

In [46]:
balanced_oof_probabilities = cross_val_predict(
    balanced_logistic_pipeline,
    X_train,
    y_train,
    cv=cross_validation,
    method="predict_proba"
)[:, 1]

In [47]:
precision_values, recall_values, thresholds = precision_recall_curve(
    y_train,
    balanced_oof_probabilities
)

In [48]:
f1_values = (
    2
    * precision_values[:-1]
    * recall_values[:-1]
    / (
        precision_values[:-1]
        + recall_values[:-1]
        + 1e-10
    )
)

best_threshold_index = np.argmax(f1_values)

best_threshold = thresholds[best_threshold_index]
best_precision = precision_values[best_threshold_index]
best_recall = recall_values[best_threshold_index]
best_f1 = f1_values[best_threshold_index]

print("Best threshold:", round(best_threshold, 3))
print("Precision:", round(best_precision, 3))
print("Recall:", round(best_recall, 3))
print("F1:", round(best_f1, 3))

Best threshold: 0.145
Precision: 0.231
Recall: 0.943
F1: 0.371


In [49]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds_to_test = [
    0.145,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60
]

threshold_results = []

for threshold in thresholds_to_test:

    predictions = (
        balanced_oof_probabilities >= threshold
    ).astype(int)

    threshold_results.append(
        {
            "Threshold": threshold,
            "Customers_Flagged": predictions.sum(),
            "Precision": precision_score(
                y_train,
                predictions,
                zero_division=0
            ),
            "Recall": recall_score(
                y_train,
                predictions,
                zero_division=0
            ),
            "F1": f1_score(
                y_train,
                predictions,
                zero_division=0
            )
        }
    )

threshold_comparison = pd.DataFrame(
    threshold_results
)

threshold_comparison.round(3)

,Threshold,Customers_Flagged,Precision,Recall,F1
0,0.145,359,0.228,0.932,0.367
1,0.200,325,0.209,0.773,0.329
2,0.300,263,0.205,0.614,0.308
3,0.400,224,0.210,0.534,0.301
4,0.500,162,0.210,0.386,0.272
5,0.600,111,0.216,0.273,0.241


In [50]:
balanced_logistic_pipeline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](33,)","['industry','country','referral_source',...,'has_satisfaction_score', 'escalation_count','days_since_last_support_ticket']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,33
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default

In [51]:
final_test_predictions = (
    balanced_logistic_pipeline.predict(X_test)
)

final_test_probabilities = (
    balanced_logistic_pipeline.predict_proba(X_test)[:, 1]
)

In [52]:
final_accuracy = accuracy_score(
    y_test,
    final_test_predictions
)

final_precision = precision_score(
    y_test,
    final_test_predictions,
    zero_division=0
)

final_recall = recall_score(
    y_test,
    final_test_predictions,
    zero_division=0
)

final_f1 = f1_score(
    y_test,
    final_test_predictions,
    zero_division=0
)

final_roc_auc = roc_auc_score(
    y_test,
    final_test_probabilities
)

final_confusion_matrix = confusion_matrix(
    y_test,
    final_test_predictions
)

print("FINAL TEST RESULTS")
print("Accuracy:", round(final_accuracy, 3))
print("Precision:", round(final_precision, 3))
print("Recall:", round(final_recall, 3))
print("F1:", round(final_f1, 3))
print("ROC-AUC:", round(final_roc_auc, 3))

print("\nConfusion Matrix:")
print(final_confusion_matrix)

FINAL TEST RESULTS
Accuracy: 0.64
Precision: 0.325
Recall: 0.591
F1: 0.419
ROC-AUC: 0.605

Confusion Matrix:
[[51 27]
 [ 9 13]]


In [53]:
fitted_preprocessor = (
    balanced_logistic_pipeline
    .named_steps["preprocessor"]
)

transformed_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

print(
    "Number of transformed features:",
    len(transformed_feature_names)
)

print("\nFirst 20 transformed features:")
print(transformed_feature_names[:20])

Number of transformed features: 52

First 20 transformed features:
['numeric__customer_tenure_days' 'numeric__subscription_count'
 'numeric__ever_upgraded' 'numeric__ever_downgraded'
 'numeric__latest_seats' 'numeric__seat_change' 'numeric__latest_mrr'
 'numeric__latest_is_trial' 'numeric__latest_auto_renew'
 'numeric__plan_changed' 'numeric__usage_event_count'
 'numeric__total_usage_count' 'numeric__unique_features_used'
 'numeric__active_usage_days' 'numeric__beta_usage_events'
 'numeric__errors_per_100_usage' 'numeric__average_duration_per_event'
 'numeric__average_usage_per_event' 'numeric__days_since_last_usage'
 'numeric__ticket_count']


In [54]:
fitted_logistic_model = (
    balanced_logistic_pipeline
    .named_steps["model"]
)

model_coefficients = (
    fitted_logistic_model
    .coef_[0]
)

print(
    "Number of coefficients:",
    len(model_coefficients)
)

Number of coefficients: 52


In [55]:
feature_importance = pd.DataFrame(
    {
        "feature": transformed_feature_names,
        "coefficient": model_coefficients
    }
)

feature_importance["absolute_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance = (
    feature_importance
    .sort_values(
        by="absolute_coefficient",
        ascending=False
    )
)

feature_importance.head(15)

,feature,coefficient,absolute_coefficient
32,categorical__country_AU,-1.085052,1.085052
34,categorical__country_DE,0.600115,0.600115
28,categorical__industry_DevTools,0.549972,0.549972
35,categorical__country_FR,0.476071,0.476071
40,categorical__referral_source_event,0.460996,0.460996
29,categorical__industry_EdTech,-0.418866,0.418866
12,numeric__unique_features_used,0.392827,0.392827
13,numeric__active_usage_days,0.372195,0.372195
15,numeric__errors_per_100_usage,-0.361437,0.361437
1,numeric__subscription_count,-0.344712,0.344712


In [56]:
top_churn_features = (
    feature_importance[
        feature_importance["coefficient"] > 0
    ]
    .sort_values(
        by="coefficient",
        ascending=False
    )
    .head(10)
)

top_churn_features[
    ["feature", "coefficient"]
]

,feature,coefficient
34,categorical__country_DE,0.600115
28,categorical__industry_DevTools,0.549972
35,categorical__country_FR,0.476071
40,categorical__referral_source_event,0.460996
12,numeric__unique_features_used,0.392827
13,numeric__active_usage_days,0.372195
48,categorical__latest_plan_tier_Enterprise,0.335174
9,numeric__plan_changed,0.307523
38,categorical__country_US,0.287226
31,categorical__industry_HealthTech,0.172947


In [57]:
top_retention_features = (
    feature_importance[
        feature_importance["coefficient"] < 0
    ]
    .sort_values(
        by="coefficient",
        ascending=True
    )
    .head(10)
)

top_retention_features[
    ["feature", "coefficient"]
]

,feature,coefficient
32,categorical__country_AU,-1.085052
29,categorical__industry_EdTech,-0.418866
15,numeric__errors_per_100_usage,-0.361437
1,numeric__subscription_count,-0.344712
18,numeric__days_since_last_usage,-0.339449
27,categorical__industry_Cybersecurity,-0.314464
43,categorical__referral_source_partner,-0.302061
49,categorical__latest_plan_tier_Pro,-0.254371
41,categorical__referral_source_organic,-0.246684
33,categorical__country_CA,-0.186098


In [58]:
all_customer_probabilities = (
    balanced_logistic_pipeline
    .predict_proba(X)[:, 1]
)

In [59]:
customer_risk_scores = pd.DataFrame(
    {
        "account_id": customer_features["account_id"],
        "churn_probability": all_customer_probabilities
    }
)

In [60]:
customer_risk_scores["churn_probability_percent"] = (
    customer_risk_scores["churn_probability"]
    * 100
).round(1)

In [61]:
customer_risk_scores.head(10)

,account_id,churn_probability,churn_probability_percent
0,A-2e4581,0.155553,15.6
1,A-43a9e3,0.592455,59.2
2,A-0a282f,0.621370,62.1
3,A-1f0ac7,0.358598,35.9
4,A-ce550d,0.549032,54.9
5,A-1b9609,0.302777,30.3
6,A-a0ca4e,0.689654,69.0
7,A-e5d6ab,0.188957,18.9
8,A-7dacce,0.477240,47.7
9,A-10b8da,0.676193,67.6


In [62]:
def assign_risk_level(churn_score):

    if churn_score >= 0.60:
        return "High"

    elif churn_score >= 0.40:
        return "Medium"

    else:
        return "Low"

In [63]:
customer_risk_scores["risk_level"] = (
    customer_risk_scores["churn_probability"]
    .apply(assign_risk_level)
)

In [64]:
customer_risk_scores.head(10)

,account_id,churn_probability,churn_probability_percent,risk_level
0,A-2e4581,0.155553,15.6,Low
1,A-43a9e3,0.592455,59.2,Medium
2,A-0a282f,0.621370,62.1,High
3,A-1f0ac7,0.358598,35.9,Low
4,A-ce550d,0.549032,54.9,Medium
5,A-1b9609,0.302777,30.3,Low
6,A-a0ca4e,0.689654,69.0,High
7,A-e5d6ab,0.188957,18.9,Low
8,A-7dacce,0.477240,47.7,Medium
9,A-10b8da,0.676193,67.6,High


In [65]:
risk_distribution = (
    customer_risk_scores["risk_level"]
    .value_counts()
)

print(risk_distribution)

risk_level
Low       199
Medium    168
High      133
Name: count, dtype: int64


In [66]:
risk_distribution_percent = (
    customer_risk_scores["risk_level"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

print(risk_distribution_percent)

risk_level
Low       39.8
Medium    33.6
High      26.6
Name: proportion, dtype: float64


In [67]:
customer_risk_scores = (
    customer_risk_scores
    .sort_values(
        by="churn_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

In [68]:
customer_risk_scores["health_score"] = (
    100
    - customer_risk_scores["churn_probability_percent"]
).round(1)

In [69]:
customer_risk_scores.head(10)

,account_id,churn_probability,churn_probability_percent,risk_level,health_score
0,A-019782,0.905537,90.6,High,9.4
1,A-5a184f,0.888714,88.9,High,11.1
2,A-1ac5e0,0.882700,88.3,High,11.7
3,A-3c1a3f,0.880406,88.0,High,12.0
4,A-32fb14,0.878306,87.8,High,12.2
5,A-271f46,0.868219,86.8,High,13.2
6,A-d922bf,0.850558,85.1,High,14.9
7,A-6c093d,0.844761,84.5,High,15.5
8,A-cd458b,0.838885,83.9,High,16.1
9,A-7f4db3,0.838663,83.9,High,16.1


In [70]:
def assign_action_priority(risk_level):

    if risk_level == "High":
        return "Priority 1 - Review"

    elif risk_level == "Medium":
        return "Priority 2 - Monitor"

    else:
        return "Priority 3 - Routine"

In [71]:
customer_risk_scores["action_priority"] = (
    customer_risk_scores["risk_level"]
    .apply(assign_action_priority)
)

In [72]:
def recommend_action(risk_level):

    if risk_level == "High":
        return (
            "Prioritise Customer Success review and "
            "investigate usage, subscription and support signals."
        )

    elif risk_level == "Medium":
        return (
            "Monitor the account and review for emerging "
            "engagement or subscription risks."
        )

    else:
        return (
            "Continue routine monitoring and maintain "
            "normal customer engagement."
        )

In [73]:
customer_health_table = customer_risk_scores.merge(
    customer_features,
    on="account_id",
    how="left"
)

In [76]:
print(customer_risk_scores.columns.tolist())

['account_id', 'churn_probability', 'churn_probability_percent', 'risk_level', 'health_score', 'action_priority']


In [77]:
def recommend_action(risk_level):

    if risk_level == "High":
        return (
            "Prioritise Customer Success review and "
            "investigate usage, subscription and support signals."
        )

    elif risk_level == "Medium":
        return (
            "Monitor the account and review for emerging "
            "engagement or subscription risks."
        )

    else:
        return (
            "Continue routine monitoring and maintain "
            "normal customer engagement."
        )

In [78]:
customer_risk_scores["recommended_action"] = (
    customer_risk_scores["risk_level"]
    .apply(recommend_action)
)

In [79]:
customer_risk_scores["risk_level"]

0      High
1      High
2      High
3      High
4      High
       ... 
495     Low
496     Low
497     Low
498     Low
499     Low
Name: risk_level, Length: 500, dtype: str

In [81]:
customer_risk_scores[
    [
        "account_id",
        "risk_level",
        "recommended_action"
    ]
].head()

,account_id,risk_level,recommended_action
0,A-019782,High,Prioritise Customer Success review and investi...
1,A-5a184f,High,Prioritise Customer Success review and investi...
2,A-1ac5e0,High,Prioritise Customer Success review and investi...
3,A-3c1a3f,High,Prioritise Customer Success review and investi...
4,A-32fb14,High,Prioritise Customer Success review and investi...


In [82]:
customer_health_table = customer_risk_scores.merge(
    customer_features,
    on="account_id",
    how="left"
)

In [83]:
customer_health_table[
    [
        "account_id",
        "industry",
        "latest_plan_tier",
        "latest_mrr",
        "churn_probability_percent",
        "health_score",
        "risk_level",
        "action_priority",
        "recommended_action"
    ]
].head(10)

,account_id,industry,latest_plan_tier,latest_mrr,churn_probability_percent,health_score,risk_level,action_priority,recommended_action
0,A-019782,DevTools,Enterprise,0,90.6,9.4,High,Priority 1 - Review,Prioritise Customer Success review and investi...
1,A-5a184f,DevTools,Basic,684,88.9,11.1,High,Priority 1 - Review,Prioritise Customer Success review and investi...
2,A-1ac5e0,DevTools,Enterprise,0,88.3,11.7,High,Priority 1 - Review,Prioritise Customer Success review and investi...
3,A-3c1a3f,DevTools,Basic,133,88.0,12.0,High,Priority 1 - Review,Prioritise Customer Success review and investi...
4,A-32fb14,DevTools,Enterprise,5174,87.8,12.2,High,Priority 1 - Review,Prioritise Customer Success review and investi...
5,A-271f46,DevTools,Enterprise,9154,86.8,13.2,High,Priority 1 - Review,Prioritise Customer Success review and investi...
6,A-d922bf,HealthTech,Enterprise,1791,85.1,14.9,High,Priority 1 - Review,Prioritise Customer Success review and investi...
7,A-6c093d,DevTools,Enterprise,0,84.5,15.5,High,Priority 1 - Review,Prioritise Customer Success review and investi...
8,A-cd458b,DevTools,Enterprise,1990,83.9,16.1,High,Priority 1 - Review,Prioritise Customer Success review and investi...
9,A-7f4db3,FinTech,Enterprise,4776,83.9,16.1,High,Priority 1 - Review,Prioritise Customer Success review and investi...


In [84]:
import numpy as np

fitted_preprocessor = (
    balanced_logistic_pipeline
    .named_steps["preprocessor"]
)

fitted_model = (
    balanced_logistic_pipeline
    .named_steps["model"]
)

transformed_customer_features = (
    fitted_preprocessor.transform(X)
)

transformed_feature_names = (
    fitted_preprocessor.get_feature_names_out()
)

model_coefficients = fitted_model.coef_[0]

In [85]:
if hasattr(
    transformed_customer_features,
    "toarray"
):
    transformed_customer_features = (
        transformed_customer_features.toarray()
    )

feature_contributions = (
    transformed_customer_features
    * model_coefficients
)

In [86]:
def get_top_risk_signals(
    contribution_row,
    feature_names,
    top_n=3
):

    positive_indexes = np.where(
        contribution_row > 0
    )[0]

    if len(positive_indexes) == 0:
        return []

    sorted_indexes = positive_indexes[
        np.argsort(
            contribution_row[positive_indexes]
        )[::-1]
    ]

    top_indexes = sorted_indexes[:top_n]

    return [
        feature_names[index]
        for index in top_indexes
    ]

In [87]:
customer_risk_signals = [
    get_top_risk_signals(
        contribution_row,
        transformed_feature_names
    )
    for contribution_row
    in feature_contributions
]

In [88]:
customer_signal_table = pd.DataFrame(
    {
        "account_id":
            customer_features["account_id"],

        "top_risk_signals":
            customer_risk_signals
    }
)

In [89]:
customer_health_table = (
    customer_health_table.merge(
        customer_signal_table,
        on="account_id",
        how="left"
    )
)

In [90]:
customer_health_table[
    [
        "account_id",
        "industry",
        "latest_plan_tier",
        "churn_probability_percent",
        "health_score",
        "risk_level",
        "top_risk_signals",
        "action_priority"
    ]
].head(10)

,account_id,industry,latest_plan_tier,churn_probability_percent,health_score,risk_level,top_risk_signals,action_priority
0,A-019782,DevTools,Enterprise,90.6,9.4,High,"[categorical__industry_DevTools, categorical__...",Priority 1 - Review
1,A-5a184f,DevTools,Basic,88.9,11.1,High,"[categorical__country_DE, categorical__industr...",Priority 1 - Review
2,A-1ac5e0,DevTools,Enterprise,88.3,11.7,High,"[categorical__industry_DevTools, categorical__...",Priority 1 - Review
3,A-3c1a3f,DevTools,Basic,88.0,12.0,High,"[categorical__industry_DevTools, categorical__...",Priority 1 - Review
4,A-32fb14,DevTools,Enterprise,87.8,12.2,High,"[categorical__industry_DevTools, categorical__...",Priority 1 - Review
5,A-271f46,DevTools,Enterprise,86.8,13.2,High,"[categorical__industry_DevTools, categorical__...",Priority 1 - Review
6,A-d922bf,HealthTech,Enterprise,85.1,14.9,High,"[categorical__latest_plan_tier_Enterprise, num...",Priority 1 - Review
7,A-6c093d,DevTools,Enterprise,84.5,15.5,High,"[categorical__industry_DevTools, numeric__acti...",Priority 1 - Review
8,A-cd458b,DevTools,Enterprise,83.9,16.1,High,"[numeric__active_usage_days, categorical__indu...",Priority 1 - Review
9,A-7f4db3,FinTech,Enterprise,83.9,16.1,High,"[numeric__active_usage_days, numeric__unique_f...",Priority 1 - Review


In [91]:
customer_health_output_path = (
    processed_data_path / "customer_health_scores.csv"
)

customer_health_table.to_csv(
    customer_health_output_path,
    index=False
)

print("Saved to:", customer_health_output_path)
print("Shape:", customer_health_table.shape)

Saved to: c:\Users\tOBESky\Documents\ML_BigDATA\saas-customer-churn-prediction\data\processed\customer_health_scores.csv
Shape: (500, 48)


In [92]:
import joblib

models_path = project_root / "models"

models_path.mkdir(
    parents=True,
    exist_ok=True
)

model_output_path = (
    models_path
    / "balanced_logistic_churn_pipeline.joblib"
)

joblib.dump(
    balanced_logistic_pipeline,
    model_output_path
)

print("Model saved to:", model_output_path)

Model saved to: c:\Users\tOBESky\Documents\ML_BigDATA\saas-customer-churn-prediction\models\balanced_logistic_churn_pipeline.joblib


In [93]:
%pip install mlflow

   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   --------- ------------------------------ 2.9/11.6 MB 15.2 MB/s eta 0:00:01
   ----------------------- ---------------- 6.8/11.6 MB 16.8 MB/s eta 0:00:01
   ------------------------------------- -- 10.7/11.6 MB 17.2 MB/s eta 0:00:01
   ---------------------------------------- 11.6/11.6 MB 16.4 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------  3.7/3.7 MB 18.1 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 17.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 14.2 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -------------------------------------- - 3.7/3.8 MB 18.2 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 17.6 MB/s  0:00:00
   ---------------------------------

In [94]:
import mlflow

In [95]:
mlflow.set_experiment(
    "saas_customer_churn_model_comparison"
)

2026/09/04 11:43:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/04 11:43:50 INFO mlflow.store.db.utils: Updating database tables
2026/09/04 11:43:53 INFO mlflow.tracking.fluent: Experiment with name 'saas_customer_churn_model_comparison' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/tOBESky/Documents/ML_BigDATA/saas-customer-churn-prediction/notebooks/mlruns/1', creation_time=1788518633396, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788518633396, lifecycle_stage='active', name='saas_customer_churn_model_comparison', tags={}, trace_location=None, workspace='default'>

In [96]:
model_results_for_mlflow = [
    {
        "model": "Logistic Regression",
        "accuracy": 0.747,
        "precision": 0.255,
        "recall": 0.080,
        "f1": 0.119,
        "roc_auc": 0.474
    },
    {
        "model": "Balanced Logistic Regression",
        "accuracy": 0.545,
        "precision": 0.210,
        "recall": 0.387,
        "f1": 0.271,
        "roc_auc": 0.484
    },
    {
        "model": "Random Forest",
        "accuracy": 0.745,
        "precision": 0.254,
        "recall": 0.080,
        "f1": 0.119,
        "roc_auc": 0.413
    },
    {
        "model": "XGBoost",
        "accuracy": 0.662,
        "precision": 0.193,
        "recall": 0.171,
        "f1": 0.178,
        "roc_auc": 0.455
    }
]

In [97]:
for result in model_results_for_mlflow:

    with mlflow.start_run(
        run_name=result["model"]
    ):

        mlflow.log_param(
            "model_name",
            result["model"]
        )

        mlflow.log_metric(
            "accuracy",
            result["accuracy"]
        )

        mlflow.log_metric(
            "precision",
            result["precision"]
        )

        mlflow.log_metric(
            "recall",
            result["recall"]
        )

        mlflow.log_metric(
            "f1",
            result["f1"]
        )

        mlflow.log_metric(
            "roc_auc",
            result["roc_auc"]
        )

In [99]:
experiment = mlflow.get_experiment_by_name(
    "saas_customer_churn_model_comparison"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

runs[
    [
        "tags.mlflow.runName",
        "metrics.accuracy",
        "metrics.precision",
        "metrics.recall",
        "metrics.f1",
        "metrics.roc_auc"
    ]
]

,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1,metrics.roc_auc
0,XGBoost,0.662,0.193,0.171,0.178,0.455
1,Random Forest,0.745,0.254,0.080,0.119,0.413
2,Balanced Logistic Regression,0.545,0.210,0.387,0.271,0.484
3,Logistic Regression,0.747,0.255,0.080,0.119,0.474


In [100]:
%pip install streamlit

   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   ------------- -------------------------- 3.7/10.5 MB 18.2 MB/s eta 0:00:01
   ----------------------------- ---------- 7.9/10.5 MB 18.7 MB/s eta 0:00:01
   ---------------------------------------- 10.5/10.5 MB 16.4 MB/s  0:00:00
   ---------------------------------------- 0.0/797.6 kB ? eta -:--:--
   ---------------------------------------- 797.6/797.6 kB 6.9 MB/s  0:00:00
   ---------------------------------------- 0.0/11.4 MB ? eta -:--:--
   ------------ --------------------------- 3.7/11.4 MB 18.1 MB/s eta 0:00:01
   ------------------------- -------------- 7.3/11.4 MB 18.1 MB/s eta 0:00:01
   ---------------------------------------  11.3/11.4 MB 18.1 MB/s eta 0:00:01
   ---------------------------------------- 11.4/11.4 MB 16.6 MB/s  0:00:00

   ----------------------------------------  0/12 [websockets]
   ----------------------------------------  0/12 [websockets]
   ---------------------------------------

In [103]:
import json

sample_customer = (
    customer_features[feature_columns]
    .dropna()
    .iloc[0]
)

sample_payload = {
    "features": json.loads(
        sample_customer.to_json()
    )
}

print(
    json.dumps(
        sample_payload,
        indent=2
    )
)

{
  "features": {
    "industry": "EdTech",
    "country": "US",
    "referral_source": "partner",
    "initial_plan_tier": "Basic",
    "customer_tenure_days": 76,
    "subscription_count": 10,
    "ever_upgraded": 1,
    "ever_downgraded": 0,
    "latest_plan_tier": "Basic",
    "latest_seats": 44,
    "seat_change": 35,
    "latest_mrr": 836,
    "latest_is_trial": 0,
    "latest_billing_frequency": "monthly",
    "latest_auto_renew": 1,
    "plan_changed": 0,
    "usage_event_count": 55,
    "total_usage_count": 535,
    "unique_features_used": 27,
    "active_usage_days": 55,
    "beta_usage_events": 4,
    "errors_per_100_usage": 7.1028037383,
    "average_duration_per_event": 2769.8,
    "average_usage_per_event": 9.7272727273,
    "days_since_last_usage": 21,
    "ticket_count": 2.0,
    "has_support_tickets": 1,
    "average_first_response_minutes": 91.0,
    "average_resolution_hours": 23.0,
    "average_satisfaction_score": 3.0,
    "has_satisfaction_score": 1,
    "escalati